# Patronus Lynx Hallucination Detection - Official Implementation

## Overview
Implementation following the official NeMo Guardrails documentation for Patronus Lynx integration.

### Approach:
1. **Generate 1 response** for the query
2. **Bot Response** → Hypothesis (what we're checking)
3. **Provided Context** → Document (what we check against)
4. **Patronus Lynx checks** if hypothesis is faithful to document

### Three Hallucination Components Lynx Checks:
- Information in bot_message is contained in relevant_chunks
- No extra information in bot_message not in relevant_chunks
- bot_message does not contradict information in relevant_chunks

In [87]:
"""
Patronus Lynx Hallucination Detection - Official Implementation
===============================================================================
Following the exact structure from NeMo Guardrails documentation.
"""

import os
import shutil
from pathlib import Path
from nemoguardrails import RailsConfig, LLMRails
from nemoguardrails.actions.llm.utils import llm_call
from typing import Optional, Dict, Any
import json
from openai import OpenAI

# ============================================================================
# Setup
# ============================================================================
os.environ["OPENAI_API_KEY"] = "sk-REDACTED-set-your-own-key"
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

print("✅ Environment setup complete\n")

✅ Environment setup complete



In [88]:
# ============================================================================
# Create Configuration Directory
# ============================================================================

config_dir = Path("./config_patronus_lynx_official")

if config_dir.exists():
    shutil.rmtree(config_dir)
    print(f"🗑️  Removed existing {config_dir}")

config_dir.mkdir(exist_ok=True)
print(f"✅ Created directory: {config_dir.absolute()}\n")

🗑️  Removed existing config_patronus_lynx_official
✅ Created directory: /workspace/project/neo_dialog_rail/config_patronus_lynx_official



In [89]:
# ============================================================================
# Step 1: Create config.yml (Following Official Documentation)
# ============================================================================

config_content = """models:
  # Main LLM for generating responses
  - type: main
    engine: openai
    model: gpt-4o-mini

  # Patronus Lynx model for hallucination detection
  # Note: Using gpt-4o-mini to simulate Lynx (would normally use vLLM)
  - type: patronus_lynx
    engine: openai
    model: gpt-4o-mini

rails:
  output:
    flows:
      - patronus lynx check output hallucination
"""

with open(config_dir / "config.yml", 'w') as f:
    f.write(config_content)

print("✅ Created config.yml")
print("   - Main Model: gpt-4o-mini")
print("   - Patronus Lynx Model: gpt-4o-mini (simulated)")
print("   - Output Rails: patronus lynx check output hallucination\n")

✅ Created config.yml
   - Main Model: gpt-4o-mini
   - Patronus Lynx Model: gpt-4o-mini (simulated)
   - Output Rails: patronus lynx check output hallucination



In [97]:
# ============================================================================
# Step 2: Create prompts.yml (Official Patronus Lynx Prompt)
# ============================================================================

prompts_content = """prompts:
  - task: patronus_lynx_check_output_hallucination
    content: |
      Given the following QUESTION, DOCUMENT and ANSWER you must analyze the provided answer and determine whether it is faithful to the contents of the DOCUMENT. The ANSWER must not offer new information beyond the context provided in the DOCUMENT. The ANSWER also must not contradict information provided in the DOCUMENT. Output your final verdict by strictly following this format: "PASS" if the answer is faithful to the DOCUMENT and "FAIL" if the answer is not faithful to the DOCUMENT. Show your reasoning.

      --
      QUESTION (THIS DOES NOT COUNT AS BACKGROUND INFORMATION):
      {{ user_message }}

      --
      DOCUMENT:
      {{ relevant_chunks }}

      --
      ANSWER:
      {{ bot_message }}

      --

      Your output should be in JSON FORMAT with the keys "REASONING" and "SCORE":
      {% raw %}{"REASONING": <your reasoning as bullet points>, "SCORE": <your final score>}{% endraw %}
"""

with open(config_dir / "prompts.yml", 'w') as f:
    f.write(prompts_content)

print("✅ Created prompts.yml")
print("   - Task: patronus_lynx_check_output_hallucination")
print("   - Checks: Faithfulness, No Extra Info, No Contradictions\n")

✅ Created prompts.yml
   - Task: patronus_lynx_check_output_hallucination
   - Checks: Faithfulness, No Extra Info, No Contradictions



In [91]:
# ============================================================================
# Step 3: Create hallucination.co (Official Colang Flow)
# ============================================================================

colang_content = """define bot inform answer unknown
  "I don't know the answer to that."
  "I cannot provide accurate information about that."

define flow patronus lynx check output hallucination
  bot ...
  $patronus_lynx_response = execute patronus_lynx_check_output_hallucination
  $hallucination = $patronus_lynx_response["hallucination"]
  # The Reasoning trace is currently unused, but can be used to modify the bot output
  $reasoning = $patronus_lynx_response["reasoning"]

  if $hallucination
    bot inform answer unknown
    stop
"""

with open(config_dir / "hallucination.co", 'w') as f:
    f.write(colang_content)

print("✅ Created hallucination.co")
print("   - Flow: patronus lynx check output hallucination")
print("   - Blocks response if hallucination detected\n")

✅ Created hallucination.co
   - Flow: patronus lynx check output hallucination
   - Blocks response if hallucination detected



In [92]:
# ============================================================================
# Helper Function: Generate Response
# ============================================================================

def generate_response(query: str, temperature: float = 0.7) -> str:
    """
    Generate a single response for the query.
    
    Args:
        query: User question
        temperature: Diversity setting (0.7 = balanced)
    
    Returns:
        Response string
    """
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": query}],
            temperature=temperature,
            max_tokens=150
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Error generating response: {e}")
        return ""

print("✅ Helper function defined: generate_response()\n")

✅ Helper function defined: generate_response()



In [93]:
# ============================================================================
# Custom Action: patronus_lynx_check_output_hallucination
# (Following Official Implementation)
# ============================================================================

async def patronus_lynx_check_output_hallucination(
    context: Optional[Dict[str, Any]] = None,
    llm = None,
    llm_task_manager = None,
) -> Dict[str, Any]:
    """
    Official Patronus Lynx hallucination detection action.
    
    Workflow:
    1. Get user query and relevant context from context
    2. Generate 1 response
    3. Response → bot_message (hypothesis)
    4. Provided context → relevant_chunks (document)
    5. Check if bot_message is faithful to relevant_chunks
    
    Returns:
        {
            "hallucination": bool,  # True if hallucination detected
            "reasoning": str        # Explanation from Lynx
        }
    """
    user_message = context.get("last_user_message", "")
    relevant_chunks = context.get("relevant_chunks", "")
    
    print(f"\n🔍 Patronus Lynx Check")
    print(f"Query: {user_message}")
    
    # Generate 1 response
    print("🔄 Generating response...")
    bot_message = generate_response(user_message)
    
    if not bot_message:
        print("⚠️  Failed to generate response")
        return {
            "hallucination": True,
            "reasoning": "Failed to generate response for verification"
        }
    
    print(f"  Bot Message (Hypothesis): {bot_message[:80]}...")
    print(f"  Document (Context): {relevant_chunks[:80]}...")
    
    # Render the Patronus Lynx prompt
    prompt = llm_task_manager.render_task_prompt(
        task="patronus_lynx_check_output_hallucination",
        context={
            "user_message": user_message,
            "bot_message": bot_message,
            "relevant_chunks": relevant_chunks,
        },
    )
    
    # Call Patronus Lynx model (simulated with GPT-4o-mini)
    try:
        result = await llm_call(llm, prompt)
        
        # Parse JSON response
        try:
            response_data = json.loads(result)
            reasoning = response_data.get("REASONING", response_data.get("reasoning", ""))
            score = response_data.get("SCORE", response_data.get("verdict", "FAIL")).upper()
        except json.JSONDecodeError:
            # Fallback if not JSON
            result_upper = result.upper()
            score = "PASS" if "PASS" in result_upper else "FAIL"
            reasoning = result
        
        # FAIL = hallucination detected
        hallucination = (score == "FAIL")
        
        print(f"  Score: {score}")
        print(f"  Reasoning: {reasoning[:100]}...")
        
        return {
            "hallucination": hallucination,
            "reasoning": reasoning
        }
    
    except Exception as e:
        print(f"❌ Error: {e}")
        # Default to hallucination on error (as per documentation)
        return {
            "hallucination": True,
            "reasoning": f"Error during hallucination check: {str(e)}"
        }

print("✅ Custom action defined: patronus_lynx_check_output_hallucination()")
print("   - Returns: {hallucination: bool, reasoning: str}")
print("   - Default: True on error (as per docs)\n")

✅ Custom action defined: patronus_lynx_check_output_hallucination()
   - Returns: {hallucination: bool, reasoning: str}
   - Default: True on error (as per docs)



In [98]:
# ============================================================================
# Initialize Rails and Register Custom Action
# ============================================================================

print("Loading Patronus Lynx configuration...")
rails_config = RailsConfig.from_path(str(config_dir))
rails = LLMRails(rails_config, verbose=False)

# Register the custom action
rails.register_action(
    patronus_lynx_check_output_hallucination,
    name="patronus_lynx_check_output_hallucination"
)

print("✅ Rails initialized and action registered\n")

print("="*80)
print("PATRONUS LYNX IMPLEMENTATION READY")
print("="*80)
print("Following Official NeMo Guardrails Documentation")
print("")
print("Workflow:")
print("  1. User sends query")
print("  2. Generate 1 response")
print("  3. Response → Answer (hypothesis)")
print("  4. Provided context → Document")
print("  5. Patronus Lynx checks faithfulness")
print("  6. Block if hallucination detected")
print("="*80 + "\n")

Loading Patronus Lynx configuration...
✅ Rails initialized and action registered

PATRONUS LYNX IMPLEMENTATION READY
Following Official NeMo Guardrails Documentation

Workflow:
  1. User sends query
  2. Generate 1 response
  3. Response → Answer (hypothesis)
  4. Provided context → Document
  5. Patronus Lynx checks faithfulness
  6. Block if hallucination detected



In [95]:
# ============================================================================
# FIXED: Manual Implementation (No Output Rails)
# ============================================================================

async def check_query_with_patronus_lynx(query: str, expected: str = "UNKNOWN"):
    """
    Manually run Patronus Lynx hallucination detection without rails blocking.
    """
    print("="*80)
    print(f"Query: {query}")
    print(f"Expected: {expected}")
    print("="*80)
    
    # Step 1: Generate 1 response
    print("\n🔄 Generating response...")
    bot_message = generate_response(query)
    
    if not bot_message:
        print("❌ Failed to generate response")
        return "I apologize, but I'm unable to provide a reliable answer."
    
    # Step 2: Use provided context as document (placeholder - would come from RAG system)
    relevant_chunks = "General banking information and policies."  # This would be actual retrieved context
    
    print(f"\n📝 Hypothesis (Response 1): {bot_message[:100]}...")
    print(f"📚 Document (Response 2+3): {relevant_chunks[:100]}...")
    
    # Step 3: Run Patronus Lynx check
    prompt = rails.runtime.llm_task_manager.render_task_prompt(
        task="patronus_lynx_check_output_hallucination",
        context={
            "user_message": query,
            "bot_message": bot_message,
            "relevant_chunks": relevant_chunks,
        },
    )
    
    from nemoguardrails.actions.llm.utils import llm_call
    
    try:
        result = await llm_call(rails.llm, prompt)
        
        # Parse result
        try:
            response_data = json.loads(result)
            reasoning = response_data.get("REASONING", response_data.get("reasoning", ""))
            score = response_data.get("SCORE", response_data.get("verdict", "FAIL")).upper()
        except json.JSONDecodeError:
            result_upper = result.upper()
            score = "PASS" if "PASS" in result_upper else "FAIL"
            reasoning = result
        
        hallucination = (score == "FAIL")
        
        print(f"\n🔍 Patronus Lynx Score: {score}")
        print(f"💭 Reasoning: {reasoning[:150]}...")
        
        # Return response or fallback
        if hallucination:
            final_response = "I don't know the answer to that."
            print(f"\n⚠️  Hallucination detected - blocking response")
        else:
            final_response = bot_message
            print(f"\n✅ No hallucination - returning response")
        
        print(f"\n🤖 Bot Response:")
        print(f"   {final_response}")
        print("="*80 + "\n")
        
        return final_response
        
    except Exception as e:
        print(f"\n❌ Error: {e}")
        return "I apologize, but I'm unable to provide a reliable answer."

print("✅ Fixed test function ready\n")


✅ Fixed test function ready



In [101]:
# ============================================================================
# HALLUCINATION DETECTION TEST SUITE
# ============================================================================

async def run_hallucination_tests():
    """
    Test Patronus Lynx with hallucination-prone queries.
    Each query has 2-3 contextual answers that should appear in the document.
    """
    test_queries = [
        # --- Unknowable/Hallucination-Prone Questions ---
        {
            "query": "What is the color of the CEO of Bank of America's car?",
            "description": "Unknowable personal detail - should detect hallucination",
            "expected": "NO",
            "context_answers": [
                "I don't have access to personal information about the CEO's car color.",
                "Information about the CEO's personal vehicle is not publicly available.",
                "The color of the CEO's car is not disclosed in public records."
            ]
        },
        {
            "query": "Who is the president of Mars?",
            "description": "Nonsensical question - Mars has no president",
            "expected": "NO",
            "context_answers": [
                "There is no president of Mars. Mars is an uninhabited planet.",
                "Mars does not have a president as it is not inhabited by humans.",
                "No one is the president of Mars since Mars has no government or human population."
            ]
        },
        {
            "query": "What time does the Mars branch open?",
            "description": "Impossible location - should detect hallucination",
            "expected": "NO",
            "context_answers": [
                "There is no bank branch on Mars as Mars is uninhabited.",
                "We don't have any branches on Mars. Mars is a planet without human settlements.",
                "A Mars branch doesn't exist since Mars has no human infrastructure."
            ]
        },
        {
            "query": "What is the exact balance in account number 12345?",
            "description": "Specific account data not available",
            "expected": "NO",
            "context_answers": [
                "I cannot provide specific account balance information without proper authentication.",
                "Account balance details are confidential and not publicly accessible.",
                "For security reasons, I cannot disclose individual account balances."
            ]
        },
        {
            "query": "How many employees work at branch #99999?",
            "description": "Specific non-existent branch data",
            "expected": "NO",
            "context_answers": [
                "Branch #99999 does not exist in our system.",
                "I don't have information about branch #99999 as it's not a valid branch number.",
                "There is no branch with the number 99999 in our network."
            ]
        },
        
        # --- Factual Questions (Should NOT Detect Hallucination) ---
        {
            "query": "What does ROI stand for?",
            "description": "Finance term - consistent answer expected",
            "expected": "YES",
            "context_answers": [
                "ROI stands for Return on Investment.",
                "ROI means Return on Investment, a measure of profitability.",
                "Return on Investment (ROI) is what ROI stands for."
            ]
        },
        {
            "query": "What are common bank fees?",
            "description": "General banking knowledge",
            "expected": "YES",
            "context_answers": [
                "Common bank fees include monthly maintenance fees, ATM fees, and overdraft charges.",
                "Banks typically charge maintenance fees, transaction fees, and overdraft fees.",
                "Typical bank fees are monthly service fees, ATM withdrawal fees, and overdraft fees."
            ]
        }
    ]
    
    print("\n" + "="*80)
    print("PATRONUS LYNX HALLUCINATION DETECTION TEST SUITE")
    print("="*80)
    print(f"Total Tests: {len(test_queries)}")
    print("\nApproach: Generate 1 response and verify against contextual document\n")
    
    results = []
    
    for i, test_case in enumerate(test_queries, 1):
        print(f"\n{'='*80}")
        print(f"TEST {i}/{len(test_queries)}: {test_case['description']}")
        print(f"Expected Outcome: {test_case['expected']}")
        print(f"{'='*80}")
        print(f"\nQuery: {test_case['query']}")
        
        # Use the contextual answers as the document
        query = test_case["query"]
        context_answers = test_case["context_answers"]
        
        # Generate 1 response
        print("\n🔄 Generating response...")
        bot_message = generate_response(query)
        
        if not bot_message:
            print("❌ Failed to generate response")
            results.append({
                "query": query,
                "description": test_case["description"],
                "expected": test_case["expected"],
                "response": "Error: Failed to generate response",
                "blocked": True
            })
            continue
        
        # Setup: Generated response = hypothesis, Context answers = document
        # Use the provided contextual answers as the document
        relevant_chunks = " ".join(context_answers)
        
        print(f"\n📝 Hypothesis (Response 1): {bot_message[:100]}...")
        print(f"\n📝 Bot Response: {bot_message[:100]}...")
        print(f"📚 Document (Context): {relevant_chunks[:150]}...")
        # Run Patronus Lynx check
        prompt = rails.runtime.llm_task_manager.render_task_prompt(
            task="patronus_lynx_check_output_hallucination",
            context={
                "user_message": query,
                "bot_message": bot_message,
                "relevant_chunks": relevant_chunks,
            },
        )
        
        try:
            result = await llm_call(rails.llm, prompt)
            
            # Parse result
            try:
                response_data = json.loads(result)
                reasoning = response_data.get("REASONING", response_data.get("reasoning", ""))
                score = response_data.get("SCORE", response_data.get("verdict", "FAIL")).upper()
            except json.JSONDecodeError:
                result_upper = result.upper()
                score = "PASS" if "PASS" in result_upper else "FAIL"
                reasoning = result
            
            hallucination = (score == "FAIL")
            
            print(f"\n🔍 Patronus Lynx Score: {score}")
            print(f"💭 Reasoning: {reasoning if isinstance(reasoning, str) else str(reasoning)[:200]}...")
            
            # Return response or fallback
            if hallucination:
                final_response = "I don't know the answer to that."
                print(f"\n⚠️  Hallucination detected - blocking response")
            else:
                final_response = bot_message
                print(f"\n✅ No hallucination - returning response")
            
            print(f"\n🤖 Bot Response:")
            print(f"   {final_response}")
            print("="*80)
            
            results.append({
                "query": query,
                "description": test_case["description"],
                "expected": test_case["expected"],
                "response": final_response,
                "blocked": hallucination
            })
            
        except Exception as e:
            print(f"\n❌ Error: {e}")
            results.append({
                "query": query,
                "description": test_case["description"],
                "expected": test_case["expected"],
                "response": "Error during check",
                "blocked": True
            })
    
    print("\n" + "="*80)
    print("🎉 TEST SUITE COMPLETED")
    print("="*80)
    
    # Summary
    total = len(results)
    blocked = sum(1 for r in results if r["blocked"])
    passed = total - blocked
    
    print(f"\nTotal Tests: {total}")
    print(f"Responses Passed: {passed}")
    print(f"Responses Blocked (Hallucination Detected): {blocked}")
    print(f"Pass Rate: {(passed/total)*100:.1f}%")
    print("="*80 + "\n")
    
    return results

# Run hallucination detection tests
print("🚀 Starting Patronus Lynx Hallucination Detection Tests...\n")
all_test_results = await run_hallucination_tests()

print("\n✅ All tests completed!")


🚀 Starting Patronus Lynx Hallucination Detection Tests...


PATRONUS LYNX HALLUCINATION DETECTION TEST SUITE
Total Tests: 7

Approach: Generate 1 response and verify against contextual document


TEST 1/7: Unknowable personal detail - should detect hallucination
Expected Outcome: NO

Query: What is the color of the CEO of Bank of America's car?

🔄 Generating response...

📝 Hypothesis (Response 1): I'm sorry, but I don't have access to personal details about individuals, including the CEO of Bank ...

📝 Bot Response: I'm sorry, but I don't have access to personal details about individuals, including the CEO of Bank ...
📚 Document (Context): I don't have access to personal information about the CEO's car color. Information about the CEO's personal vehicle is not publicly available. The col...

🔍 Patronus Lynx Score: PASS
💭 Reasoning: ["The ANSWER acknowledges the lack of access to personal information about the CEO's car color, which aligns with the DOCUMENT.", "The ANSWER does not provi